In [1]:
################################################################################
#  Evaluate LEDSNet using the test set
################################################################################

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import h5py

# -------------------------------------------------------------------------
# 1.  Device
# -------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------------------------------
# 2.  Paths
# -------------------------------------------------------------------------
directory    = '/Test_Dataset/'
test_h5_name = 'MITBIHAR_test_set_only.h5'   # <-- the file you created earlier
model_path   = os.path.join(directory, 'Trained_Model',
                            'LMSNet_Trained_parameters.pth')

# -------------------------------------------------------------------------
# 3.  Load the saved test set
# -------------------------------------------------------------------------
test_h5_path = os.path.join(directory, test_h5_name)
print("Loading test set from:", test_h5_path)

with h5py.File(test_h5_path, 'r') as h5f:
    test_data   = h5f['ECG_Signals'][:]
    test_labels = h5f['ECG_Labels'][:]

# Ensure labels are 1D ints
test_labels = np.asarray(test_labels).astype(np.int64).squeeze()

# Ensure shape is (N, 1, 77)
test_data = np.asarray(test_data).astype(np.float32)
if test_data.ndim == 2:            # (N, 77) -> (N, 1, 77)
    test_data = np.expand_dims(test_data, axis=1)
elif test_data.ndim == 3:
    # assume already (N, 1, 77)
    pass
else:
    raise ValueError(f"Unexpected ECG_Signals shape: {test_data.shape}")

# Build DataLoader
batch_size = 4096
test_tensor_x = torch.from_numpy(test_data)     
test_tensor_y = torch.from_numpy(test_labels)   
test_loader   = DataLoader(TensorDataset(test_tensor_x, test_tensor_y),
                           batch_size=batch_size, shuffle=False)

# -------------------------------------------------------------------------
# 4.  Define the LEDSNet model architecture
# -------------------------------------------------------------------------
class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.depthwise = nn.Conv1d(in_channels, in_channels, kernel_size,
                                   stride=stride, padding=padding,
                                   groups=in_channels, bias=False)
        self.pointwise = nn.Conv1d(in_channels, out_channels, kernel_size=1,
                                   bias=False)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.relu(x)

class FireModule(nn.Module):
    def __init__(self, in_channels, squeeze_channels, expand1x1_channels, expand3x3_channels):
        super().__init__()
        self.squeeze = nn.Sequential(
            nn.Conv1d(in_channels, squeeze_channels, kernel_size=1),
            nn.BatchNorm1d(squeeze_channels),
            nn.ReLU(inplace=True)
        )
        self.expand1x1 = nn.Sequential(
            nn.Conv1d(squeeze_channels, expand1x1_channels, kernel_size=1),
            nn.BatchNorm1d(expand1x1_channels),
            nn.ReLU(inplace=True)
        )
        self.expand3x3 = nn.Sequential(
            nn.Conv1d(squeeze_channels, expand3x3_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(expand3x3_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        x = self.squeeze(x)
        return torch.cat([self.expand1x1(x), self.expand3x3(x)], 1)

class CombinedModel(nn.Module):
    def __init__(self, num_classes=16):
        super().__init__()
        self.stages = nn.ModuleDict({
            'Stage 1': nn.Sequential(
                nn.Conv1d(1, 64, kernel_size=5, stride=4, padding=1, bias=False),
                nn.BatchNorm1d(64),
                nn.ReLU(inplace=True)
            ),
            'Stage 2': DepthwiseSeparableConv1d(64, 32, kernel_size=5, stride=1, padding=1),
            'Stage 3': DepthwiseSeparableConv1d(32, 16, kernel_size=5, stride=4, padding=1),
            'Stage 4': DepthwiseSeparableConv1d(16, 16, kernel_size=5, stride=1, padding=1),
            'Stage 5': FireModule(16, 32, 16, 16),
            'Stage 6': FireModule(32, 64, 8, 8),
            'Stage 7': nn.AdaptiveAvgPool1d(1),
            'Stage 8': nn.Flatten(),
            'Stage 9': nn.Linear(16, num_classes)
        })
    def forward(self, x):
        for stage in self.stages.values():
            x = stage(x)
        return x

# -------------------------------------------------------------------------
# 5.  Load the trained weights
# -------------------------------------------------------------------------
model = CombinedModel(num_classes=16).to(device)
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print("Loaded trained model from:", model_path)

# -------------------------------------------------------------------------
# 6.  Evaluate on test set
# -------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
test_loss, correct, total = 0.0, 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        loss = criterion(outputs, y)
        test_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

print("\n================ TEST RESULTS ================")
print(f"Average loss : {test_loss/len(test_loader):.4f}")
print(f"Accuracy     : {100.0*correct/total:.2f}%")
print("Confusion matrix:\n", confusion_matrix(all_labels, all_preds))
print("Classification report:\n", classification_report(all_labels, all_preds))
print("================================================")


Using device: cuda
Loading test set from: /scratch/user/uqabulbu/Data/MITBIHAR_test_set_only.h5
Loaded trained model from: /scratch/user/uqabulbu/Data/Output_Models/LMSNet_05_47kp_16c_GAN_Ep_1k_1DS_1000_parameters.pth

================ TEST RESULTS ================
Average loss : 0.0855
Accuracy     : 98.58%
Confusion matrix:
 [[22272     7    10    30     0    75     0    19     3     9     3     1
      0     1     0     0]
 [    3  2406     0    10     0     0     0     0     1     0     0     1
      0     0     0     1]
 [    4     0  2164     2     0     6     0     0     0     0     0     0
      0     0     0     0]
 [   33    13     3  2066     1     2     0    14     3     0     1     1
      0     0     0     0]
 [    0     0     0     0  1086     0     0     0     0     0     0     0
      0     0     0     0]
 [   63     3     5     1     0   685     0     0     0     6     0     1
      0     0     0     0]
 [    2     0     0     0     0     0    76     0     0     0    